In [1]:
# Task 3.1
import pymongo
client = pymongo.MongoClient("127.0.0.1",27017)
db = client['electromart']
product_coll = db['product']
customer_coll = db['customer']
order_coll = db['order']
print(client.database_names()) # all the databases in mongodb server
print(db.collection_names())  # all the collections in electromart
product_coll.delete_many({}) # Remove all documents
customer_coll.delete_many({}) # Remove all documents
order_coll.delete_many({}) # Remove all documents
client.close()

['OUTLETS', 'admin', 'demo', 'electromart', 'entertainment', 'jp_mobile', 'local']
['order', 'customer', 'product']


c:\program files\python37\lib\site-packages\ipykernel_launcher.py:8: DeprecationWarning: database_names is deprecated. Use list_database_names instead.
  
c:\program files\python37\lib\site-packages\ipykernel_launcher.py:9: DeprecationWarning: collection_names is deprecated. Use list_collection_names instead.
  if __name__ == "__main__":


In [2]:
# Task 3.2
def import_data():
    import json
    file = open("products.json",'r')
    product_data = json.load(file)
    file.close()
    for product in product_data:
        if product["price"] > 0: # check for positive number
            product_coll.insert_one(product)
    
    file = open("customers.json",'r')
    customer_data = json.load(file)
    file.close()
    rejected_customers = list() # store the customers with duplicated email
    #check for duplicate email address, if no dup, then can insert into db
    for customer in customer_data:
        if customer_coll.find_one({"email": customer['email'] }) is None:
            customer_coll.insert_one(customer)
        else: 
            rejected_customers.append(customer)
    print("Rejected customers:")
#     print(rejected_customers)
    for doc in rejected_customers:
        print(doc)

import_data() #run function

Rejected customers:
{'customer_id': 'C603', 'name': 'Wei Chen Too', 'email': 'wei.chen@example.com', 'address': '123 Main Street'}
{'customer_id': 'C703', 'name': 'Siti Two', 'email': 'siti.nurhaliza@example.com', 'address': '45 Jasmine Avenue'}


In [3]:
# Task 3.3

def place_order (customer_id, product_id, quantity):
    import datetime
    product_ordered = product_coll.find_one({"product_id": product_id})
    if product_ordered is None: # Product not in collection
        print("Product not available.")
        return #exit function
    if product_ordered["stock"] < quantity:
        print("InsufficientStockError")
        return #exit function
    #insert product into order collection
    order_made = {
        "order_id" : 'O' + str(datetime.datetime.now()),
        "customer_id" : customer_id,
        "timestamp" : str(datetime.datetime.now()),
        "product_id" : product_id,
        "quantity" : quantity
    }
    order_coll.insert_one(order_made)
    
    #Deduct quantity from product stock number
    product_coll.update_one(
        {"product_id" : product_id},
        {"$inc" : {"stock": -quantity} }
    )
    print("Order place successfully.")

place_order("C201", "P111", 2)
place_order("C301", "P444", 5)
place_order("C401", "P1012", 15)
place_order("C201", "P444", 1)
place_order("C301", "P222", 1)
place_order("C501", "P999", 3)

Order place successfully.
Order place successfully.
InsufficientStockError
Order place successfully.
Order place successfully.
Order place successfully.


In [4]:
# Task 3.4
def generate_sales_report():
    # Total revenue per category
    category_revenue = {}

    for order in order_coll.find():
        product = product_coll.find_one({"product_id": order["product_id"]})
        category = product["category"]
        revenue = product["price"] * order["quantity"]

        if category in category_revenue:
            category_revenue[category] += revenue
        else:
            category_revenue[category] = revenue

    # Total orders per customer
    customer_orders = {}

    for customer in customer_coll.find():
        customer_id = customer["customer_id"]
        count = order_coll.count_documents({"customer_id": customer_id})
        customer_orders[customer_id] = count

    # Display
    print("\nSales Report")
    print("{:20}{:<10}".format("Category","Revenue"))
    for category, revenue in category_revenue.items():
        print("{:20}{:<10}".format(category, round(revenue,2)))

    print("\n{:30}{:10}".format("Customer","Total Orders"))
    for customer_id, count in customer_orders.items():
        customer = customer_coll.find_one({"customer_id": customer_id})
        print("{:30}{:<10}".format(customer["name"], count))

generate_sales_report()


Sales Report
Category            Revenue   
Electronics         1599.98   
Audio               1019.91   
Computers           1299.99   

Customer                      Total Orders
Wei Chen                      2         
Li Mei                        2         
Kwok Ho                       0         
Siti Aminah                   1         
Rahul Raj                     0         
Yuan Zhang                    0         
Hsin Liu                      0         
Wai Yee Ng                    0         
Wei Jie Tan                   0         
Aisyah Binti Abdullah         0         
Budi Santoso                  0         
Xiao Ming                     0         
Chia-Hui Wu                   0         
Kai Yip                       0         
Siti Nurhaliza                0         
Subramaniam Raju              0         
Adinda Putri                  0         
Hong Sheng                    0         
Mei Xiu                       0         
Ahmad Hakim                   0        

In [5]:
# Task 3.5

def get_products_by_category(category):
    result = product_coll.find({"category": category})

    for product in result:
        print(product)
        
get_products_by_category("Electronics")

{'_id': ObjectId('69995b5f5b190516dd6fd805'), 'product_id': 'P111', 'name': 'Smartphone X', 'category': 'Electronics', 'price': 799.99, 'stock': 98}
{'_id': ObjectId('69995b5f5b190516dd6fd809'), 'product_id': 'P555', 'name': 'Tablet Pro', 'category': 'Electronics', 'price': 499.99, 'stock': 80}


In [6]:
# Task 3.6
def find_customer_by_email(email):
    customer = customer_coll.find_one({"email": email})

    if customer:
        print(customer)
    else:
        print("Customer not found")
        
find_customer_by_email("wei.chen@example.com")

{'_id': ObjectId('69995b5f5b190516dd6fd818'), 'customer_id': 'C201', 'name': 'Wei Chen', 'email': 'wei.chen@example.com', 'address': '123 Main Street'}


In [7]:
# Task 3.7
def get_orders_by_customer(customer_id):
    result = order_coll.find({"customer_id": customer_id})

    for order in result:
        print(order)

get_orders_by_customer("C201")

{'_id': ObjectId('69995b605b190516dd6fd82c'), 'order_id': 'O2026-02-21 15:14:40.003733', 'customer_id': 'C201', 'timestamp': '2026-02-21 15:14:40.003733', 'product_id': 'P111', 'quantity': 2}
{'_id': ObjectId('69995b605b190516dd6fd82e'), 'order_id': 'O2026-02-21 15:14:40.012945', 'customer_id': 'C201', 'timestamp': '2026-02-21 15:14:40.012945', 'product_id': 'P444', 'quantity': 1}


In [8]:
# Task 3.8
def menu():
    while True:
        print("\nElectroMart Inventory Management System")
        print("1. Import Data")
        print("2. Place Order")
        print("3. Generate Sales Report")
        print("4. Get Products by Category")
        print("5. Find Customer by Email")
        print("6. Get Orders by Customer")
        print("7. Exit")

        choice = input("Enter choice: ")

        if choice == "1":
            import_data()

        elif choice == "2":
            customer_id = input("Customer ID: ")
            product_id = input("Product ID: ")
            quantity = int(input("Quantity: "))
            place_order(customer_id, product_id, quantity)

        elif choice == "3":
            generate_sales_report()

        elif choice == "4":
            category = input("Category: ")
            get_products_by_category(category)

        elif choice == "5":
            email = input("Email: ")
            find_customer_by_email(email)

        elif choice == "6":
            customer_id = input("Customer ID: ")
            get_orders_by_customer(customer_id)

        elif choice == "7":
            print("Exiting program")
            break

        else:
            print("Invalid option")

menu()
client.close()


ElectroMart Inventory Management System
1. Import Data
2. Place Order
3. Generate Sales Report
4. Get Products by Category
5. Find Customer by Email
6. Get Orders by Customer
7. Exit
Enter choice: 3

Sales Report
Category            Revenue   
Electronics         1599.98   
Audio               1019.91   
Computers           1299.99   

Customer                      Total Orders
Wei Chen                      2         
Li Mei                        2         
Kwok Ho                       0         
Siti Aminah                   1         
Rahul Raj                     0         
Yuan Zhang                    0         
Hsin Liu                      0         
Wai Yee Ng                    0         
Wei Jie Tan                   0         
Aisyah Binti Abdullah         0         
Budi Santoso                  0         
Xiao Ming                     0         
Chia-Hui Wu                   0         
Kai Yip                       0         
Siti Nurhaliza                0         
Sub

In [21]:
# Task 3.9

import pymongo
client = pymongo.MongoClient("127.0.0.1",27017)
db = client['electromart']
order_coll = db['order']

for doc in order_coll.find({}):
    items = [
        {   'product_id': doc.get('product_id'),
            'quantity': doc.get('quantity')
        }
    ]
    search = {'_id': doc['_id']}
    update_data = {'$set': {'items': items}, '$unset': {'product_id':0,'quantity':0}}
    order_coll.update_one(search, update_data)

client.close()

In [29]:
# Task 3.10
def place_order(customer_id, items):
    # Iterate and check product is available and quantity is sufficient
    for item in items:
        product_id = item['product_id']
        quantity   = item['quantity']
        
        product = product_coll.find_one({"product_id": product_id})
        
        if product is None: # No such product
            print(f"Product {product_id} is not available.")
            return
        
        if product['stock'] < quantity:
            print("InsufficientStockError")
            return
    
    # Product available and quantity sufficient, therefore insert product into order collection
    import datetime
    order_made = {
        "order_id" : 'O' + str(datetime.datetime.now()),
        "customer_id" : customer_id,
        "timestamp" : str(datetime.datetime.now()),
        "items" : items
    }
    order_coll.insert_one(order_made)
    
    #Deduct quantity from product stock number
    for item in items:
        product_coll.update_one(
            {"product_id" : item['product_id']},
            {"$inc" : {"stock": -item['quantity']} }
        )
    
    print("Order place successfully.")
    
# Testing code
place_order("C201", [{"product_id": "P111", "quantity": 2}, {"product_id": "P444", "quantity": 1}])

In [37]:
# Task 3.11
def get_orders_by_customer (customer_id):
    customer_orders = order_coll.find({"customer_id":customer_id})
    
    for order in customer_orders:
        print("Order ID:", order.get('order_id'))
        print("Timestamp:", order.get('timestamp'))
        print("Items:")
        
        for item in order.get('items',[]): # default empty list
            product_id = item.get('product_id')
            quantity   = item.get('quantity')
            print(f"Product ID: {product_id}, Quantity: {quantity}")
    

In [38]:
get_orders_by_customer("C201")

Order ID: O2026-02-21 15:14:40.003733
Timestamp: 2026-02-21 15:14:40.003733
Items:
Product ID: None, Quantity: None
Order ID: O2026-02-21 15:14:40.012945
Timestamp: 2026-02-21 15:14:40.012945
Items:
Product ID: None, Quantity: None
Order ID: O2026-02-21 17:32:27.514751
Timestamp: 2026-02-21 17:32:27.514751
Items:
Product ID: P111, Quantity: 2
Product ID: P444, Quantity: 1
Order ID: O2026-02-21 17:32:53.099846
Timestamp: 2026-02-21 17:32:53.099846
Items:
Product ID: P111, Quantity: 2
Product ID: P444, Quantity: 1
Order ID: O2026-02-21 17:33:21.206853
Timestamp: 2026-02-21 17:33:21.206853
Items:
Product ID: P111, Quantity: 2
Product ID: P444, Quantity: 1
